In [1]:
!pip install -q streamlit transformers accelerate "bitsandbytes>=0.46.1" peft pandas torch
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O cloudflared.deb
!dpkg -i cloudflared.deb || apt-get -f install -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.5 MB/s eta 0:00:00
Selecting previously unselected package cloudflared.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.5.0) ...
Setting up cloudflared (2026.5.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.environ["BASE_MODEL"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["ADAPTER_PATH"] = "/content/drive/MyDrive/lora_model_8bit"
os.environ["QUANT_MODE"] = "8bit"

In [3]:
%%writefile app.py
import os
import re
import json
from pathlib import Path

import pandas as pd
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = os.getenv("BASE_MODEL", "Qwen/Qwen2.5-3B-Instruct")
ADAPTER_PATH = os.getenv("ADAPTER_PATH", "")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "1024"))
QUANT_MODE = os.getenv("QUANT_MODE", "8bit")

st.set_page_config(page_title="Interview Coding Assistant", layout="wide")
st.title("Автоматическая разметка интервью")
st.markdown("---")

SYSTEM_PROMPT = """Ты — эксперт по анализу интервью. Твоя задача — выделить тематические коды в тексте интервью и выполнить разметку.

**Формат вывода:**
**Общий код N: Название общего кода**
"цитата из интервью" - **конкретный код**

**Важные правила:**
1. Используй ТОЛЬКО русский язык
2. Ответ должен содержать только коды и цитаты, без лишних слов и повторений
3. Каждый общий код должен содержать минимум 2-3 конкретных кода с цитатами
4. Цитаты должны быть точными фрагментами из интервью
5. Сохраняй оригинальную пунктуацию внутри цитат
6. Не добавляй пояснений до или после разметки

Теперь выполни разметку для следующего интервью."""

def build_prompt(topic: str, transcript: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Тема интервью:\n{topic}\n\n"
        f"Текст интервью:\n{transcript}\n\n"
        f"Выполни разметку интервью.<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

@st.cache_resource(show_spinner="Загрузка модели, подождите...")
def load_model():
    try:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id

        if QUANT_MODE == "4bit":
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
        else:
            bnb_cfg = BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_threshold=6.0,
                llm_int8_has_fp16_weight=False,
            )

        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_cfg,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )

        if ADAPTER_PATH and Path(ADAPTER_PATH).exists():
            model = PeftModel.from_pretrained(base, ADAPTER_PATH, is_trainable=False)
            st.success(f"Адаптер загружен: {ADAPTER_PATH}")

        model.eval()
        return tokenizer, model

    except Exception as e:
        st.error(f"Ошибка загрузки модели: {str(e)}")
        raise

def generate(
    tokenizer,
    model,
    topic: str,
    transcript: str,
    max_new_tokens: int,
    num_beams: int,
    repetition_penalty: float,
    no_repeat_ngram_size: int
) -> str:
    prompt = build_prompt(topic.strip(), transcript.strip())

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=num_beams,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    generated = generated.split("<|im_end|>")[0].strip()

    if generated.startswith("system"):
        generated = ""
    elif generated.startswith("user"):
        generated = ""

    return generated

def parse_output(raw_text: str) -> list:
    rows = []
    current_general = ""

    lines = raw_text.split('\n')

    for line in lines:
        line = line.strip()
        if not line:
            continue

        general_pattern = r'\*{0,2}Общий код\s*\d+\s*[:\-]\s*(.+?)\*{0,2}$'
        match_general = re.match(general_pattern, line, re.IGNORECASE)

        if match_general:
            current_general = match_general.group(1).strip().strip('*').strip()
            continue

        quote_pattern = r'^["“«](.+?)["”»]\s*[-–—]\s*\*{0,2}(.+?)\*{0,2}$'
        match_quote = re.match(quote_pattern, line)

        if match_quote and current_general:
            rows.append({
                "Общий код": current_general,
                "Конкретный код": match_quote.group(2).strip(),
                "Цитата": match_quote.group(1).strip(),
            })
        elif current_general and (line.startswith('"') or line.startswith('“')):
            quote = line.strip('"“»').strip()
            if quote:
                rows.append({
                    "Общий код": current_general,
                    "Конкретный код": "",
                    "Цитата": quote,
                })
    return rows

def validate_parsed_data(rows: list) -> bool:
    if not rows:
        return False

    for row in rows:
        if not row.get("Общий код") or not row.get("Цитата"):
            return False

    return True

with st.sidebar:
    st.header("Параметры генерации")

    num_beams = st.slider(
        "num_beams",
        min_value=1,
        max_value=8,
        value=5,
        help="Количество лучей при beam search"
    )

    repetition_penalty = st.slider(
        "repetition_penalty",
        min_value=1.0,
        max_value=2.0,
        value=1.5,
        step=0.1,
        help="Штраф за повторения (больше = меньше повторений)"
    )

    no_repeat_ngram_size = st.slider(
        "no_repeat_ngram_size",
        min_value=0,
        max_value=8,
        value=4,
        help="Размер n-граммы, которую нельзя повторять"
    )

    max_new_tokens = st.slider(
        "max_new_tokens",
        min_value=256,
        max_value=2048,
        value=MAX_NEW_TOKENS,
        step=64,
        help="Максимальное количество генерируемых токенов"
    )

    st.divider()

    st.subheader("Информация о модели")
    st.caption(f"**Модель:** {BASE_MODEL.split('/')[-1]}")
    st.caption(f"**Квантизация:** {QUANT_MODE}")
    st.caption(f"**Адаптер:** {'загружен' if ADAPTER_PATH and Path(ADAPTER_PATH).exists() else 'не используется'}")

    st.divider()
    st.markdown("""
    **Инструкция:**
    1. Введите тему исследования
    2. Вставьте транскрипт интервью
    3. Нажмите "Разметить"
    4. Скачайте результат в CSV или JSON
    """)

topic = st.text_area(
    "Тема исследования",
    height=100,
    placeholder="Например: Поколенческая идентичность, жизненные выборы и устойчивость российской молодежи",
    help="Укажите основную тему интервью"
)

transcript = st.text_area(
    "Текст интервью",
    height=400,
    placeholder="Вставьте транскрипт интервью сюда...\n\nИнтервьюер: Вопрос...\nИнформант: Ответ...",
    help="Вставьте полный транскрипт интервью с репликами"
)

col1, col2, col3 = st.columns([1, 1, 4])
run_btn = col1.button("Разметить", type="primary", use_container_width=True)
clear_btn = col2.button("Очистить", use_container_width=True)

if clear_btn:
    st.rerun()
try:
    tokenizer, model = load_model()
except Exception as e:
    st.stop()

if run_btn:
    if not topic.strip():
        st.error("Укажите тему исследования.")
    elif not transcript.strip():
        st.error("Вставьте текст интервью.")
    else:
        with st.spinner("Генерация разметки... Это может занять несколько минут."):
            try:
                raw_output = generate(
                    tokenizer, model,
                    topic, transcript,
                    max_new_tokens, num_beams,
                    repetition_penalty, no_repeat_ngram_size,
                )

                with st.expander("Сырой ответ модели", expanded=False):
                    st.code(raw_output, language="markdown")

                st.subheader("Структурированная разметка")
                parsed_rows = parse_output(raw_output)

                if parsed_rows and validate_parsed_data(parsed_rows):
                    df = pd.DataFrame(parsed_rows)

                    st.dataframe(
                        df,
                        use_container_width=True,
                        column_config={
                            "Общий код": st.column_config.TextColumn("Общий код", width="medium"),
                            "Конкретный код": st.column_config.TextColumn("Конкретный код", width="large"),
                            "Цитата": st.column_config.TextColumn("Цитата", width="large"),
                        }
                    )

                    st.info(f"Размечено: {len(parsed_rows)} фрагментов | {df['Общий код'].nunique()} уникальных общих кодов")
                    col_dl1, col_dl2, _ = st.columns([1, 1, 2])

                    with col_dl1:
                        st.download_button(
                            "Скачать CSV",
                            df.to_csv(index=False).encode("utf-8-sig"),
                            "interview_coding.csv",
                            "text/csv",
                            help="Скачать результат в формате CSV"
                        )

                    with col_dl2:
                        st.download_button(
                            "Скачать JSON",
                            json.dumps(parsed_rows, ensure_ascii=False, indent=2).encode("utf-8"),
                            "interview_coding.json",
                            "application/json",
                            help="Скачать результат в формате JSON"
                        )
                else:
                    st.warning("Не удалось распарсить структуру ответа. Проверьте формат вывода модели.")
                    st.markdown("**Сырой ответ модели:**")
                    st.code(raw_output, language="markdown")

            except Exception as e:
                st.error(f"Ошибка при генерации: {str(e)}")
                st.info("Попробуйте уменьшить параметры генерации или сократить текст интервью.")

st.markdown("---")
st.caption("Совет: Для лучших результатов используйте структурированные транскрипты с пометками 'Интервьюер:' и 'Информант:'")

Writing app.py


In [4]:
import subprocess, time, os, signal

!pkill -f ngrok || true
!pkill -f streamlit || true

proc = subprocess.Popen([
    "streamlit", "run", "/content/app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
], stdout=open("/content/streamlit.log", "w"), stderr=open("/content/streamlit.log", "a"))

time.sleep(8)
print("streamlit pid:", proc.pid)

^C
^C
streamlit pid: 4851


In [5]:
import subprocess, time, re, os, textwrap

cloud = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=open("/content/cf.log", "w"),
    stderr=open("/content/cf.log", "a"),
)

time.sleep(8)

log = open("/content/cf.log", "r").read()
print(log[-3000:])

2026-05-21T20:57:19Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-21T20:57:19Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-21T20:57:25Z INF +--------------------------------------------------------------------------------------------+
2026-05-21T20:57:25Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-21T20:57:25Z INF |  https://great-potentially-binding-conversion.trycloud